# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template and examples for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL (FAIR^2 Croissant).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# Print summary metadata
md = dataset.metadata
print(f"{md.name}: {md.description}")
print(f"Published: {md.datePublished}\nLicense: {md.license}\nIdentifier: {md.identifier}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The Croissant schema organizes data into **record sets**, each corresponding to a data table or file. Each record set contains **fields** referencing columns. All entities are referenced by their `@id` for consistency.

In [ ]:
# Find all record sets in the dataset
record_sets = dataset.record_sets
if not record_sets:
    print("No record sets found in this schema! (Dataset may be metadata-only or reference files outside Croissant.)")
else:
    print(f"Record set @ids found: {[rs['@id'] for rs in record_sets]}")
    for rs in record_sets:
        print(f"\nRecord set: {rs['@id']}")
        print(f"  Name: {rs.get('name', rs['@id'])}")
        fields = rs.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        elif isinstance(fields, str):
            fields = [fields]
        print("  Fields and their @id:")
        for fid in fields:
            print(f"    {fid}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Record set and field `@id`s are shown above.

You should replace the values in `example_record_set_id` and `record_sets_ids` with the actual record set `@id`s found above for your exploration. Here we try the first one for demonstration.

In [ ]:
# -- Inspect which record sets are available, take the first for demo --
record_sets = dataset.record_sets
if not record_sets:
    print("No record sets to load!")
else:
    # Use the @id of all record sets
    record_sets_ids = [rs['@id'] for rs in record_sets]
    # For demonstration, pick the first
    example_record_set_id = record_sets_ids[0]

    dataframes = {}

    for record_set in record_sets_ids:
        records = list(dataset.records(record_set=record_set))
        df = pd.DataFrame(records)
        dataframes[record_set] = df
        print(f"Loaded {len(df)} rows for record set: {record_set}")
        print(f"  Columns: {df.columns.tolist()}")

    # Show the head of the first record set dataframe
    print("\nPreview first 5 rows of the first record set DataFrame:")
    print(dataframes[example_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Below, we demonstrate with one numeric field from the first record set (adjust to your data fields as found above).

In [ ]:
# Pick your record set and field @id (update as necessary)
record_set_id = example_record_set_id
df = dataframes[record_set_id]

# Display available columns/fields
print(f"Available columns in this record set:", df.columns.tolist())

# Try to select a numeric field (example: 'cr:field/coverage', adjust as needed)
possible_numeric_fields = [col for col in df.columns if df[col].dtype.kind in 'fi']
if not possible_numeric_fields:
    # No inferred numeric fields, try pick one by name
    for col in df.columns:
        if any(x in col.lower() for x in ['coverage', 'mw', 'count', 'number', 'percent', 'abundance']):
            possible_numeric_fields.append(col)

if possible_numeric_fields:
    numeric_field = possible_numeric_fields[0]
    print(f"Numeric field chosen for EDA: {numeric_field}")
else:
    print("No numeric field found for EDA. (Adjust by specifying manually)")
    numeric_field = df.columns[0]  # fallback, may not work

# Filter records with numeric_field > threshold (e.g., 10)
try:
    threshold = 10
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping (e.g., by protein description or accession)
    group_fields = [col for col in df.columns if 'desc' in col.lower() or 'accession' in col.lower() or 'type' in col.lower()]
    if group_fields:
        group_field = group_fields[0]
        print(f"Grouping by field: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(grouped_df.head())
    else:
        print("No suitable non-numeric group field found.")
except Exception as e:
    print(f"Could not perform EDA automatically: {e}")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Simple histogram and scatterplot examples are provided. You may need to adjust field names depending on your schema.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field in df.columns:
    plt.figure(figsize=(7,4))
    df[numeric_field].hist(bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # Try scatter between first two numeric fields if available
    if len(possible_numeric_fields) >= 2:
        plt.figure(figsize=(6, 4))
        plt.scatter(df[possible_numeric_fields[0]], df[possible_numeric_fields[1]], alpha=0.6)
        plt.xlabel(possible_numeric_fields[0])
        plt.ylabel(possible_numeric_fields[1])
        plt.title(f"Scatterplot of {possible_numeric_fields[0]} vs {possible_numeric_fields[1]}")
        plt.show()
else:
    print("No valid numeric field for visualization.")

## 6. Conclusion
In this notebook, we've demonstrated how to load, explore, and perform basic analysis of a dataset described by a Croissant metadata schema using the `mlcroissant` library.

- **Dataset**: Protein abundance and characteristics from mass spectrometry of human mast cell EVs
- **Source**: FAIR^2 Croissant package
- **Exploration included**: Discovering record sets/fields, extracting records, filtering, normalization, grouping, and simple visualizations

For more advanced analysis, consult the [mlcroissant documentation](https://mlcommons.github.io/croissant/) and adapt field/grouping logic to match your dataset's detailed schema.